<a href="https://colab.research.google.com/github/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/blob/main/students/seif_kanaan/code/Canadian_Disasters_Monitor_Reproducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canadian Disasters Monitor

Run all three code cells in Google Colab.

This rebuilds the maps from saved satellite exports. It does not perform a new Earth Engine computation.


In [ ]:
print("1/3 Loading the GitHub project...", flush=True)
from pathlib import Path
import os
import subprocess
REPO = 'OSS4008A---Introduction-to-Remote-Sensing-and-Applications'
REPO_DIR = Path('/content') / REPO
fresh_checkout = not (REPO_DIR / '.git').is_dir()
REPO_URL = f'https://github.com/AdityaR28/{REPO}.git'
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'}, check=True, timeout=180)
PROJECT_DIR = REPO_DIR / 'students' / 'seif_kanaan'
for folder in ('code', 'data', 'results'):
    if not (PROJECT_DIR / folder).is_dir():
        raise RuntimeError(f'Required folder is missing: {folder}')
os.chdir(PROJECT_DIR)
print('Ready: students/seif_kanaan')
print('Verified folders: code, data, results')
print('Working location:', Path.cwd())

print("2/3 Installing pinned dependencies...", flush=True)
import sys
import subprocess
import importlib
CODE_DIR = PROJECT_DIR / 'code'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(CODE_DIR / 'requirements-map.txt')], check=True, timeout=600)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
modules = ['ee', 'numpy', 'rasterio', 'PIL', 'processing.pipeline', 'processing.satellite_flood', 'processing.satellite_earthquake', 'processing.satellite_tornado', 'processing.satellite_pipeline']
for name in modules:
    importlib.import_module(name)
    print(f'Loaded: {name}')
print('\nDependency installation and import checks passed.')

print("3/3 Verifying saved satellite data...", flush=True)
import hashlib, json
MAP_DATA_DIR = PROJECT_DIR / "results" / "satellite_data"
# Verify event metadata and all satellite-layer checksums.
catalogue = json.loads((MAP_DATA_DIR / "catalogue.json").read_text())
layer_count = 0

for entry in catalogue["events"]:
    event_file = MAP_DATA_DIR / entry["url"]
    event = json.loads(event_file.read_text())

    expected_manifest = entry.get(
        "manifest_sha256",
        "092f4f6f3e08a3170c129052bbfb0262f1e40b2f34e332ca1e5ca6440ff4a332"
    )
    if hashlib.sha256(event_file.read_bytes()).hexdigest() != expected_manifest:
        raise ValueError(f"Event metadata mismatch: {entry['id']}")

    for layer in event["layers"]:
        image_file = event_file.parent / layer["url"]
        expected_layer = (
            layer.get("sha256")
            or entry["layers_sha256"][layer["url"]]
        )

        if hashlib.sha256(image_file.read_bytes()).hexdigest() != expected_layer:
            raise ValueError(f"Layer mismatch: {image_file.name}")

        layer_count += 1

    print(f"{entry['hazard']}: {len(event['layers'])} layers verified")

if len(catalogue["events"]) != 4 or layer_count != 28:
    raise ValueError("Expected four events and 28 satellite layers.")

report = {
    "status": "passed_file_integrity",
    "events": 4,
    "layers": layer_count,
    "scope": "Imported existing exports; no new satellite computation."
}
(PROJECT_DIR / "results" / "import_checks.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\nGitHub data verified: 4 events, 28 verified layers.")
print("Fresh checkout:", fresh_checkout)


In [ ]:
import folium
import json
import html

DEFAULT_LAYERS = {
    "wildfire": "post_rgb",
    "flood": "post_vv_db",
    "earthquake": "delta_vv_db",
    "tornado": "pre_rgb",
}

YEARS = {
    "wildfire": 2023,
    "flood": 2021,
    "earthquake": 2025,
    "tornado": 2025,
}

def build_case_map(entry):
    event_file = MAP_DATA_DIR / entry["url"]
    data = json.loads(event_file.read_text())
    hazard = entry["hazard"]

    if hazard == "wildfire":
        title = f"{data['event']['name']} {data['event']['year']}"
        bbox = data["event"]["aoi_bbox"]
    else:
        title = data["title"]
        bbox = data["bbox"]

    west, south, east, north = bbox
    center = [(south + north) / 2, (west + east) / 2]

    summary = data.get("summary") or (
        "Sentinel-2 before/after imagery and exploratory "
        "spectral-change indicators."
    )

    case_map = folium.Map(
        location=center, tiles=None,
        height=600, control_scale=True
    )

    folium.TileLayer(
        "Esri.WorldGrayCanvas",
        control=False,
        max_native_zoom=16,
        max_zoom=19
    ).add_to(case_map)

    for layer in data["layers"]:
        label = (
            "Candidate spectral change (exploratory)"
            if hazard == "wildfire" and layer["id"] == "candidate"
            else layer["label"]
        )

        image = folium.raster_layers.ImageOverlay(
            image=str(event_file.parent / layer["url"]),
            bounds=layer["bounds"],
            name=label,
            overlay=False,
            show=(layer["id"] == DEFAULT_LAYERS[hazard]),
            opacity=1,
            interactive=True,
            attribution=(
                f"Contains modified Copernicus Sentinel data "
                f"({YEARS[hazard]}); full credits: Event information"
            )
        )

        legend = "; ".join(
            item["label"] for item in layer.get("legend", [])
        )
        folium.Popup(
            html.escape(label) + "<br>" + html.escape(legend),
            max_width=450
        ).add_to(image)

        image.add_to(case_map)

    if hazard == "wildfire":
        folium.GeoJson(
            str(event_file.parent / data["reference"]["url"]),
            name="Historical perimeter (context)",
            style_function=lambda feature: {
                "color": "#00ffff", "weight": 2, "fillOpacity": 0
            },
            show=False,
            attribution=(
                "Contains information licensed under the "
                "Open Government Licence – British Columbia."
            )
        ).add_to(case_map)

    # Retain interpretation limits and source credits in each saved map.
    limits_html = "".join(
        f"<li>{html.escape(text)}</li>"
        for text in data.get("limitations", [])
    )

    sources_html = ""
    for source in data.get("sources", []):
        url = html.escape(source["url"], quote=True)
        label = html.escape(source["label"])
        attribution = html.escape(source.get("attribution") or "")
        license_text = html.escape(source.get("license") or "")

        sources_html += (
            f'<li><a href="{url}" target="_blank" '
            f'rel="noopener noreferrer">{label}</a>'
            f"<br>{attribution}<br>{license_text}</li>"
        )

    information = (
        f"<b>{html.escape(title)}</b>"
        f"<p>{html.escape(summary)}</p>"
        f"<b>Interpretation limits</b><ul>{limits_html}</ul>"
        f"<b>Sources and credits</b><ul>{sources_html}</ul>"
    )

    folium.Marker(
        center,
        tooltip="Event information: limits and source credits",
        icon=folium.Icon(color="blue", icon="info-sign"),
        popup=folium.Popup(
            information, max_width=520, max_height=350
        )
    ).add_to(case_map)

    case_map.fit_bounds([[south, west], [north, east]])
    folium.LayerControl(collapsed=False).add_to(case_map)

    return case_map, title, summary


catalogue = json.loads(
    (MAP_DATA_DIR / "catalogue.json").read_text()
)
MAP_EXPORT_DIR = PROJECT_DIR / "results" / "maps"
MAP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

event_maps = {}

for entry in catalogue["events"]:
    case_map, title, summary = build_case_map(entry)
    output_file = MAP_EXPORT_DIR / f"{entry['id']}_map.html"
    case_map.save(str(output_file))

    event_maps[entry["id"]] = {
        "map": case_map,
        "title": title,
        "summary": summary,
    }
    print(f"{entry['hazard']}: map saved")

print("\nAll four maps ready.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

EVENT_LABELS = {
    "wildfire": "Wildfire — McDougall Creek (2023)",
    "flood": "Flood — Lower Fraser (2021)",
    "earthquake": "Earthquake — Mount King George (2025)",
    "tornado": "Tornado — Law Lake (2025)",
}

event_selector = widgets.Dropdown(
    options=[
        (EVENT_LABELS[entry["hazard"]], entry["id"])
        for entry in catalogue["events"]
    ],
    value=catalogue["default_event"],
    description="Event:",
    layout=widgets.Layout(width="650px", max_width="100%")
)

map_output = widgets.Output()

def show_selected_event(change=None):
    current = event_maps[event_selector.value]

    with map_output:
        clear_output(wait=True)
        display(Markdown(
            f"**{current['title']}**\n\n"
            f"{current['summary']}\n\n"
            "*Exploratory observations; missing or excluded pixels "
            "remain unknown. Click the blue information marker "
            "for interpretation limits and source credits.*"
        ))
        display(current["map"])

event_selector.observe(show_selected_event, names="value")

display(event_selector, map_output)
show_selected_event()